# 11 · Performance tuning

## The golden rules
1. **Read less**: use Parquet/Delta, select only the columns you need, filter early (on partition columns if possible).
2. **Shuffle less**: broadcast small tables in joins, avoid unnecessary `distinct`/`orderBy`.
3. **Stay in the JVM**: built-in functions over UDFs.
4. **Don't pull data to the driver**: no `collect()`/`toPandas()` on big data. Use `df.isEmpty()` instead of `df.count() == 0`.
5. **Cache** only what you reuse — and unpersist when done.
6. **Watch file sizes**: aim for ~128 MB–1 GB files; thousands of tiny files are slow to read.
7. **Look at the Spark UI** for skew (one task much slower than the others) and spills.

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
import io, contextlib

def plan_of(df):
    """Return the text that df.explain() would print."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

import time
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{DATA}/products.csv", header=True, inferSchema=True)

## Caching
Because of lazy evaluation, every action **recomputes** the whole recipe from the source. If you use the
same expensive DataFrame several times, `cache()` keeps it in memory after the first computation.
`cache()` is itself lazy — the data is stored during the next action.

In [ ]:
big = spark.range(8_000_000).withColumn("k", F.col("id") % 1000).withColumn("v", F.rand(seed=1))
expensive = big.groupBy("k").agg(F.avg("v").alias("avg_v"), F.stddev("v").alias("sd"))

for i in range(2):
    t = time.time(); expensive.count(); print(f"uncached run {i}: {time.time() - t:.2f}s")

expensive.cache()
for i in range(2):
    t = time.time(); expensive.count(); print(f"cached run {i}:   {time.time() - t:.2f}s  (first run fills the cache)")

print(expensive.is_cached, expensive.storageLevel)
expensive.unpersist()
print("after unpersist:", expensive.is_cached)

`persist(StorageLevel.MEMORY_AND_DISK)` lets you choose where it's stored (`cache()` = memory, spilling to disk).
Caching isn't free (memory, and the time to fill it) — only use it for data you **reuse**.

## Reading less: partition pruning, predicate pushdown, column pruning
With Parquet, Spark can skip whole folders (**PartitionFilters**), skip chunks inside files using
min/max statistics (**PushedFilters**), and read only the needed columns (**ReadSchema**). Check the plan:

In [ ]:
orders.write.mode("overwrite").partitionBy("status").parquet(f"{OUT}/perf_orders")
po = spark.read.parquet(f"{OUT}/perf_orders")

q = po.filter((F.col("status") == "returned") & (F.col("quantity") >= 4)).select("order_id", "quantity")
q.explain()

In the `FileScan parquet` line you should see:
* `PartitionFilters: [isnotnull(status#5), (status#5 = returned)]` (the `#5` is just Spark's internal column id) → only the `status=returned/` folder is read
* `PushedFilters: [IsNotNull(quantity), GreaterThanOrEqual(quantity,4)]` → filter applied inside the reader
* `ReadSchema: struct<order_id:int,quantity:int>` → only two columns read from disk

## Joins: broadcast vs shuffle (sort-merge)

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)      # disable auto-broadcast for the demo
spark.conf.set("spark.sql.adaptive.enabled", False)
print("SortMergeJoin" in plan_of(orders.join(products, "product_id")))                 # shuffles both sides
print("BroadcastHashJoin" in plan_of(orders.join(F.broadcast(products), "product_id")))  # explicit hint wins
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.adaptive.enabled", True)

## Data skew
If one key has far more rows than the others, the task handling it takes forever while the rest sit idle.
In the Spark UI this shows up as one task with a much longer duration / bigger input than the median.

In [ ]:
skewed = spark.range(1_000_000).withColumn("key", F.when(F.col("id") < 900_000, 0).otherwise(F.col("id") % 10))
(skewed.repartition(8, "key")
       .groupBy(F.spark_partition_id().alias("partition")).count()
       .orderBy("partition").show())          # one partition gets ~90% of the data

Fixes:
* **AQE skew join** handles skewed *joins* automatically (`spark.sql.adaptive.skewJoin.enabled`, on by default).
* **Salting** for aggregations: add a random "salt" to split the hot key, aggregate in two steps.

In [ ]:
salted = (skewed.withColumn("salt", (F.rand(seed=0) * 8).cast("int"))
                .groupBy("key", "salt").agg(F.count("*").alias("partial"))    # step 1: hot key spread over 8 groups
                .groupBy("key").agg(F.sum("partial").alias("n")))             # step 2: combine
salted.orderBy("key").show(3)

## Output file sizes
Each partition writes one file. Before writing a small result use `coalesce(n)`; for big outputs,
`repartition` by a column or set `maxRecordsPerFile`.

### Exercise 1
Build a query `pruned` on `po` that keeps **completed** orders with **quantity > 3**, selecting only `order_id` and `quantity`. Then set `plan = plan_of(pruned)`. (The check confirms partition pruning and pushdown happened.)

In [ ]:
pruned = None  # TODO
plan = None    # TODO

In [ ]:
# Check — run me after your answer
assert "PartitionFilters: [isnotnull(status" in plan and "= completed)" in plan, "expected a PartitionFilter on status"
assert "GreaterThan(quantity,3)" in plan, "expected quantity > 3 pushed down"
assert "struct<order_id:int,quantity:int>" in plan, "expected only 2 columns read"
print("Correct!")

### Exercise 2
Create `cached_products`: `products` **cached and materialised** (run an action so it's actually in memory).

In [ ]:
cached_products = None  # TODO

In [ ]:
# Check — run me after your answer
assert cached_products.is_cached and cached_products.storageLevel.useMemory
print("Correct!")

### Exercise 3
Join `orders` and `products` on `product_id` **forcing a broadcast** of products, and store the plan text in `bj_plan`.

In [ ]:
bj_plan = None  # TODO

In [ ]:
# Check — run me after your answer
assert "BroadcastHashJoin" in bj_plan
print("Correct!")

### Exercise 4
Write the completed orders from `po` to `f"{OUT}/single"` as Parquet (overwrite) so that the output contains **exactly one** data file.

In [ ]:
# TODO

In [ ]:
# Check — run me after your answer
files = [f for f in os.listdir(f"{OUT}/single") if f.startswith("part-")]
assert len(files) == 1, files
assert spark.read.parquet(f"{OUT}/single").count() == 4035
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/11_performance_tuning.ipynb`](../solutions/11_performance_tuning.ipynb).